## PrISM Data Preprocessing notebook
### Step 0. Setup the paths and env variables

In [2]:
# =========================
# STEP 0 — Setup & contracts
# =========================
from pathlib import Path
import json, sys, re
import numpy as np
import pandas as pd
from tqdm import tqdm

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import _canon  # you already use this in other loaders

BASE   = ROOT / "data"
RAW    = BASE / "raw_data" / "PrISM"
CLEANED = BASE / "cleaned_premerge"

SCHEMA_PATH       = ROOT / "Unification" / "schemas" / "continuous_stream_schema.json"
ACTIVITY_MAP_PATH = ROOT / "Unification" / "schemas" / "activity_mapping.json"

SCHEMA       = json.loads(SCHEMA_PATH.read_text())
ACT_MAP_FULL = json.loads(ACTIVITY_MAP_PATH.read_text())

UNKNOWN_ID = int(ACT_MAP_FULL.get("unknown_activity_id", 9000))
TARGET_HZ  = int(SCHEMA.get("rate_hz", 50))

RAW2ID  = { _canon(k): int(v) for k, v in ACT_MAP_FULL.get("mapping", {}).items() }
ID2NAME = { int(x["id"]): x["name"] for x in ACT_MAP_FULL["label_set"] }

TASKS_DIR = RAW / "tasks"

print("RAW:", RAW)
print("TASKS_DIR:", TASKS_DIR)
print("TARGET_HZ:", TARGET_HZ)


RAW: /home/aidan/IMU_LM_Data/data/raw_data/PrISM
TASKS_DIR: /home/aidan/IMU_LM_Data/data/raw_data/PrISM/tasks
TARGET_HZ: 50


### Step 1. Ingest, preporccess and map the data 

In [2]:
# ============================
# STEP 1 — Load PRISM motion.txt (native ~50Hz) → per-session dataframe
# ============================

G_M_S2 = 9.80665

def list_tasks(tasks_dir: Path):
    return sorted([p.name for p in tasks_dir.iterdir() if p.is_dir()])

TASK_NAMES = list_tasks(TASKS_DIR)
TASK2NATIVE_ID = {t: (i + 1) for i, t in enumerate(TASK_NAMES)}

KEEP_COLS = [
    "timestamp",
    "userAcceleration.x", "userAcceleration.y", "userAcceleration.z",
    "gravity.x", "gravity.y", "gravity.z",
    "rotationRate.x", "rotationRate.y", "rotationRate.z",
]

def read_motion_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", engine="python")
    missing = [c for c in KEEP_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"[PRISM] Missing columns in {path}: {missing}")
    return df[KEEP_COLS].copy()

def parse_subject_trial(folder_name: str):
    # p10 / p2 / p1 ...
    m = re.match(r"^p(\d+)$", folder_name, flags=re.IGNORECASE)
    if m:
        return f"p{int(m.group(1))}", None

    # P12-03 (participant 12, trial 3)
    m = re.match(r"^P(\d+)-(\d+)$", folder_name)
    if m:
        return f"p{int(m.group(1))}", int(m.group(2))

    # latte_making numeric folders: 10, 11, ...
    m = re.match(r"^(\d+)$", folder_name)
    if m:
        return f"p{int(m.group(1))}", None

    # latte_making authors: 1-authors, 2-authors, ...
    m = re.match(r"^(\d+)-authors$", folder_name, flags=re.IGNORECASE)
    if m:
        return f"auth{int(m.group(1))}", None

    return None, None

def load_prism_native(tasks_dir: Path) -> pd.DataFrame:
    seen_tasks = set()
    session_counts = {}
    sessions = []

    # DEBUG COUNTERS (raw motion.txt folders vs loaded vs skipped-unparsed)
    raw_motion_dirs = {}
    loaded_dirs = {}
    skipped_unparsed = {}

    STEP_NS = int(1e9 // TARGET_HZ)  # 20_000_000 at 50Hz

    for task in tqdm(TASK_NAMES, desc="[PRISM] load tasks"):
        task_root = tasks_dir / task / "dataset" / "original"
        if not task_root.exists():
            continue

        for pdir in sorted([p for p in task_root.iterdir() if p.is_dir()], key=lambda x: x.name):
            motion_path = pdir / "motion.txt"
            if not motion_path.exists():
                continue

            # DEBUG: count raw folders that have motion.txt
            raw_motion_dirs[task] = raw_motion_dirs.get(task, 0) + 1

            subject_id, trial = parse_subject_trial(pdir.name)
            if subject_id is None:
                # DEBUG: motion exists but folder name didn't match any pattern
                skipped_unparsed[task] = skipped_unparsed.get(task, 0) + 1
                continue

            df = read_motion_file(motion_path)
            if df.empty:
                continue

            # timestamp: ms -> ns (session-relative)
            t_ms = df["timestamp"].to_numpy(np.float64)
            t0 = float(np.min(t_ms))
            ts_ns = np.round((t_ms - t0) * 1e6).astype("int64")

            # acc: (userAcc + gravity) * g
            ax = (df["userAcceleration.x"].to_numpy(np.float64) + df["gravity.x"].to_numpy(np.float64)) * G_M_S2
            ay = (df["userAcceleration.y"].to_numpy(np.float64) + df["gravity.y"].to_numpy(np.float64)) * G_M_S2
            az = (df["userAcceleration.z"].to_numpy(np.float64) + df["gravity.z"].to_numpy(np.float64)) * G_M_S2

            # gyro: rotationRate.*
            gx = df["rotationRate.x"].to_numpy(np.float64)
            gy = df["rotationRate.y"].to_numpy(np.float64)
            gz = df["rotationRate.z"].to_numpy(np.float64)

            # sanity: ~50Hz jitter ok
            d = np.diff(np.sort(ts_ns))
            d = d[d > 0]
            if d.size:
                med_ms = float(np.median(d)) / 1e6
                if not (10.0 <= med_ms <= 30.0):
                    raise ValueError(f"[PRISM] {task}/{pdir.name} median dt={med_ms:.2f}ms (expected ~20ms)")

            # resample to exact 50Hz grid
            order = np.argsort(ts_ns)
            ts_ns = ts_ns[order]
            ax, ay, az = ax[order], ay[order], az[order]
            gx, gy, gz = gx[order], gy[order], gz[order]

            end_ns = int(ts_ns[-1])
            target_ts = np.arange(0, end_ns + STEP_NS, STEP_NS, dtype=np.int64)

            ax = np.interp(target_ts, ts_ns, ax).astype("float32")
            ay = np.interp(target_ts, ts_ns, ay).astype("float32")
            az = np.interp(target_ts, ts_ns, az).astype("float32")
            gx = np.interp(target_ts, ts_ns, gx).astype("float32")
            gy = np.interp(target_ts, ts_ns, gy).astype("float32")
            gz = np.interp(target_ts, ts_ns, gz).astype("float32")

            session_id = f"{task}_{subject_id}" + (f"_t{trial:02d}" if trial is not None else "")

            out = pd.DataFrame({
                "dataset": "prism",
                "subject_id": subject_id,
                "session_id": session_id,
                "timestamp_ns": target_ts,

                "acc_x": ax,
                "acc_y": ay,
                "acc_z": az,
                "gyro_x": gx,
                "gyro_y": gy,
                "gyro_z": gz,

                "dataset_activity_id": np.int16(TASK2NATIVE_ID[task]),
                "dataset_activity_label": str(task),
            })

            sessions.append(out)
            seen_tasks.add(task)
            session_counts[task] = session_counts.get(task, 0) + 1

            # DEBUG: count folders that actually became loaded sessions
            loaded_dirs[task] = loaded_dirs.get(task, 0) + 1

    print(f"[PRISM] tasks found: {len(TASK_NAMES)} -> {TASK_NAMES}")
    print(f"[PRISM] tasks loaded: {len(seen_tasks)} -> {sorted(seen_tasks)}")
    print("[PRISM] sessions per task: " + ", ".join([f"{t}:{session_counts.get(t,0)}" for t in TASK_NAMES]))
    print(f"[PRISM] total sessions: {sum(session_counts.values())}")

    print("[PRISM] raw motion.txt dirs vs loaded vs skipped-unparsed:")
    for t in TASK_NAMES:
        rm = raw_motion_dirs.get(t, 0)
        ld = loaded_dirs.get(t, 0)
        sk = skipped_unparsed.get(t, 0)
        print(f"  {t}: raw_motion_dirs={rm}, loaded={ld}, skipped_unparsed={sk}")

    return pd.concat(sessions, ignore_index=True) if sessions else pd.DataFrame()

prism_native = load_prism_native(TASKS_DIR)
print("PRISM native rows:", len(prism_native))
prism_native.head(3)


[PRISM] load tasks: 100%|██████████| 8/8 [00:24<00:00,  3.12s/it]

[PRISM] tasks found: 8 -> ['MakeCereal', 'MakeCoffee', 'MakeSandwich', 'MakeStencil', 'MakeTea', 'cooking', 'latte_making', 'skin_care']
[PRISM] tasks loaded: 8 -> ['MakeCereal', 'MakeCoffee', 'MakeSandwich', 'MakeStencil', 'MakeTea', 'cooking', 'latte_making', 'skin_care']
[PRISM] sessions per task: MakeCereal:26, MakeCoffee:32, MakeSandwich:31, MakeStencil:20, MakeTea:33, cooking:17, latte_making:22, skin_care:5
[PRISM] total sessions: 186
[PRISM] raw motion.txt dirs vs loaded vs skipped-unparsed:
  MakeCereal: raw_motion_dirs=26, loaded=26, skipped_unparsed=0
  MakeCoffee: raw_motion_dirs=32, loaded=32, skipped_unparsed=0
  MakeSandwich: raw_motion_dirs=31, loaded=31, skipped_unparsed=0
  MakeStencil: raw_motion_dirs=20, loaded=20, skipped_unparsed=0
  MakeTea: raw_motion_dirs=33, loaded=33, skipped_unparsed=0
  cooking: raw_motion_dirs=17, loaded=17, skipped_unparsed=0
  latte_making: raw_motion_dirs=22, loaded=22, skipped_unparsed=0
  skin_care: raw_motion_dirs=5, loaded=5, skippe

,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,dataset_activity_id,dataset_activity_label
0,prism,p1,MakeCereal_p1_t01,0,2.140865,9.112032,-2.724153,0.003549,-0.149163,-0.082127,1,MakeCereal
1,prism,p1,MakeCereal_p1_t01,20000000,2.139240,9.090304,-2.737160,0.002582,-0.129329,-0.073177,1,MakeCereal
2,prism,p1,MakeCereal_p1_t01,40000000,2.097855,9.194843,-2.805918,0.041731,-0.103313,-0.086626,1,MakeCereal


### Step 2. Map the data and audit the mapping

In [3]:
# ============================
# STEP 2 — Mapping audit (native → global)
# ============================
if prism_native.empty:
    raise SystemExit("No PRISM rows after STEP 1. Check raw paths / folder structure.")

raw_counts = (
    prism_native["dataset_activity_label"]
    .astype("string")
    .map(_canon)
    .value_counts()
    .rename_axis("raw_label")
    .reset_index(name="count")
)

raw_counts["mapped_id"] = raw_counts["raw_label"].map(RAW2ID).fillna(UNKNOWN_ID).astype(int)
raw_counts["mapped_nm"] = raw_counts["mapped_id"].map(lambda x: ID2NAME.get(int(x), "other"))

unmapped = raw_counts.loc[raw_counts["mapped_id"] == UNKNOWN_ID]
print(f"[PRISM] Unique raw labels: {len(raw_counts)} | Unmapped: {len(unmapped)}")

total_ct = int(raw_counts["count"].sum())
mapped_ct = int(raw_counts.loc[raw_counts["mapped_id"] != UNKNOWN_ID, "count"].sum())
print(f"coverage={100.0*mapped_ct/max(total_ct,1):.2f}%  (mapped={mapped_ct:,}/{total_ct:,})")

if not unmapped.empty:
    print("\nUnmapped raw labels (add these to activity_mapping.json if desired):")
    print(unmapped.sort_values("count", ascending=False).to_string(index=False))

raw_counts


[PRISM] Unique raw labels: 8 | Unmapped: 0
coverage=100.00%  (mapped=3,037,272/3,037,272)


,raw_label,count,mapped_id,mapped_nm
0,cooking,602284,15,adl_food
1,maketea,529834,15,adl_food
2,makesandwich,448286,15,adl_food
3,makecoffee,406434,15,adl_food
4,latte_making,357631,15,adl_food
5,makestencil,314695,14,adl_desk_device
6,makecereal,302634,15,adl_food
7,skin_care,75474,19,adl_personal_care


### Step 3. Build and clean dataset in stream json fromat

In [4]:
# ============================
# STEP 3 — Add global_* and schema-order
# ============================
def to_continuous_stream_prism(df_native: pd.DataFrame) -> pd.DataFrame:
    if df_native.empty:
        return pd.DataFrame(columns=[c["name"] for c in SCHEMA["columns"]])

    raw_key = df_native["dataset_activity_label"].astype("string").map(_canon)
    gid     = raw_key.map(RAW2ID).fillna(UNKNOWN_ID).astype("int16")
    glabel  = gid.map(lambda x: ID2NAME.get(int(x), "other")).astype("string")

    out = pd.DataFrame({
        "dataset": df_native["dataset"].astype("string"),
        "subject_id": df_native["subject_id"].astype("string"),
        "session_id": df_native["session_id"].astype("string"),
        "timestamp_ns": df_native["timestamp_ns"].astype("int64"),

        "acc_x": df_native["acc_x"].astype("float32"),
        "acc_y": df_native["acc_y"].astype("float32"),
        "acc_z": df_native["acc_z"].astype("float32"),
        "gyro_x": df_native["gyro_x"].astype("float32"),
        "gyro_y": df_native["gyro_y"].astype("float32"),
        "gyro_z": df_native["gyro_z"].astype("float32"),

        "global_activity_id": gid,
        "global_activity_label": glabel,

        "dataset_activity_id": df_native["dataset_activity_id"].astype("int16"),
        "dataset_activity_label": df_native["dataset_activity_label"].astype("string"),
    })

    order = [c["name"] for c in SCHEMA["columns"]]
    return out[order]

prism_df = to_continuous_stream_prism(prism_native)
print("PRISM unified rows:", len(prism_df))
prism_df.head(3)


PRISM unified rows: 3037272


,dataset,subject_id,session_id,timestamp_ns,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,global_activity_id,global_activity_label,dataset_activity_id,dataset_activity_label
0,prism,p1,MakeCereal_p1_t01,0,2.140865,9.112032,-2.724153,0.003549,-0.149163,-0.082127,15,adl_food,1,MakeCereal
1,prism,p1,MakeCereal_p1_t01,20000000,2.139240,9.090304,-2.737160,0.002582,-0.129329,-0.073177,15,adl_food,1,MakeCereal
2,prism,p1,MakeCereal_p1_t01,40000000,2.097855,9.194843,-2.805918,0.041731,-0.103313,-0.086626,15,adl_food,1,MakeCereal


### Step 4. Audit check the unified frame

In [5]:
# ============================
# STEP 4 — QA
# ============================
groups = ["subject_id", "session_id"]

print("Subjects:", prism_df["subject_id"].nunique(), "| Sessions:", prism_df["session_id"].nunique())

# monotonic per group
viol = 0
for (_sid, _sess), g in prism_df.groupby(groups, sort=False):
    ts = g["timestamp_ns"].to_numpy()
    if ts.size and not np.all(np.diff(ts) >= 0):
        viol += 1
print("Monotonic violations (groups):", viol)

def est_hz_ns(ts_ns: pd.Series):
    arr = ts_ns.to_numpy()
    if arr.size < 3:
        return np.nan
    dt = np.diff(arr) / 1e9
    dt = dt[(dt > 0) & np.isfinite(dt)]
    return float(np.median(1.0 / dt)) if dt.size else np.nan

hz = prism_df.groupby(groups)["timestamp_ns"].apply(est_hz_ns)
print(f"Median Hz: {np.nanmedian(hz.values):.2f} (target={SCHEMA['rate_hz']})")

req = SCHEMA["expectations"]["required_not_null"]
pct = prism_df[req].notnull().all(axis=1).mean() * 100
print(f"Rows meeting required-not-null: {pct:.2f}%")

cov = (prism_df["global_activity_id"] != UNKNOWN_ID).mean() * 100
print(f"Global mapping coverage: {cov:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-15 dataset_activity_label:")
print(prism_df["dataset_activity_label"].value_counts().head(15))

print("\nTop-15 global labels:")
print(prism_df["global_activity_label"].value_counts().head(15))


Subjects: 31 | Sessions: 186
Monotonic violations (groups): 0
Median Hz: 50.00 (target=50)
Rows meeting required-not-null: 100.00%
Global mapping coverage: 100.0% (unknown=9000)

Top-15 dataset_activity_label:
dataset_activity_label
cooking         602284
MakeTea         529834
MakeSandwich    448286
MakeCoffee      406434
latte_making    357631
MakeStencil     314695
MakeCereal      302634
skin_care        75474
Name: count, dtype: Int64

Top-15 global labels:
global_activity_label
adl_food             2647103
adl_desk_device       314695
adl_personal_care      75474
Name: count, dtype: Int64


### Step 5. Save outputs

In [6]:
# ============================
# STEP 5 — Save
# ============================
out_path = CLEANED / "prism_clean_data.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
prism_df.to_parquet(out_path, index=False)
print("Saved →", out_path)


Saved → /home/aidan/IMU_LM_Data/data/cleaned_premerge/prism_clean_data.parquet
